# Pricing Path-Dependent Exotic Options via Monte Carlo

## 1. The Necessity of Monte Carlo for Path-Dependency

For standard European options, the terminal payoff $\Phi(S_T)$ depends exclusively on the state of the asset at maturity $T$. Because the exact probability density function of $S_T$ is known under the Geometric Brownian Motion (GBM) assumption (it is lognormally distributed), the risk-neutral expectation can be integrated analytically, yielding the Black-Scholes formula.

However, many over-the-counter (OTC) derivatives possess payoffs that are **path-dependent**, meaning their value relies on the entire historical trajectory of the asset over the interval $[0, T]$, denoted as $\{S_t\}_{t \in [0,T]}$. 

In these cases, we must simulate the joint distribution of the entire path. Let us examine two fundamental examples: Asian and Barrier options.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'font.family': 'serif'})

# Global Parameters
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.20
N_paths = 10000
M_steps = 252

def simulate_exact_paths(S0, T, r, sigma, N, M):
    """Simulates GBM paths exactly to eliminate discretization bias."""
    dt = T / M
    paths = np.zeros((M + 1, N))
    paths[0] = S0
    Z = np.random.standard_normal((M, N))
    for t in range(1, M + 1):
        paths[t] = paths[t - 1] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[t - 1])
    return paths

np.random.seed(42)
paths = simulate_exact_paths(S0, T, r, sigma, N_paths, M_steps)

## 2. Asian Options (Arithmetic Average)

Asian options have payoffs dependent on the average price of the underlying asset over a predetermined observation period. This averaging feature makes them cheaper than standard European options because the volatility of an average is lower than the volatility of the asset itself.

Consider a discrete Arithmetic Asian Call option. The payoff at maturity is:

$$\Phi_{Asian} = \max\left( \frac{1}{M} \sum_{i=1}^M S_{t_i} - K, 0 \right)$$

**Why MC is required:** The sum of lognormally distributed random variables (the asset prices at different times) is *not* lognormally distributed. Therefore, the exact distribution of the arithmetic average is unknown, and no general closed-form solution exists. (Note: *Geometric* Asian options do have closed-form solutions, but they are rarely traded).

In [3]:
def price_asian_call(paths, K, T, r):
    """
    Prices an Arithmetic Asian Call Option.
    Averages over all simulated time steps (excluding S0 typically).
    """
    # Calculate the arithmetic average path-by-path (axis=0 means average across time)
    average_prices = np.mean(paths[1:, :], axis=0)
    
    # Calculate payoffs and discount
    payoffs = np.maximum(average_prices - K, 0)
    discounted_payoffs = np.exp(-r * T) * payoffs
    
    price = np.mean(discounted_payoffs)
    std_err = np.std(discounted_payoffs) / np.sqrt(len(discounted_payoffs))
    return price, std_err

asian_price, asian_err = price_asian_call(paths, K, T, r)
print(f"Arithmetic Asian Call Price: {asian_price:.4f} (Std Err: {asian_err:.4f})")

Arithmetic Asian Call Price: 5.6690 (Std Err: 0.0781)


## 3. Barrier Options (Up-and-Out)

Barrier options are standard European options that are either activated (knock-in) or nullified (knock-out) if the underlying asset crosses a specific price level (the barrier, $B$) at any point prior to maturity.

Consider a discrete **Up-and-Out Call**. The option pays off like a normal call, *unless* the asset price breaches $B$ from below, at which point it becomes worthless. The payoff is:

$$\Phi_{UaO} = \max(S_T - K, 0) \times \mathbf{1}_{\{ \max_{t \in [0,T]} S_t < B \}}$$

where $\mathbf{1}$ is the indicator function. The discrete monitoring frequency (e.g., daily closing prices) heavily impacts the probability of hitting the barrier. Monte Carlo seamlessly handles discrete monitoring by simply checking the maximum value across the discretized steps.

In [ ]:
def price_up_and_out_call(paths, K, B, T, r):
    """
    Prices a discretely monitored Up-and-Out Call Option.
    """
    # Find the maximum price reached along each path
    path_maxima = np.max(paths, axis=0)
    
    # Determine which paths survived (never breached barrier B)
    survival_indicator = (path_maxima < B).astype(float)
    
    # Calculate European payoff and apply the knockout indicator
    terminal_prices = paths[-1, :]
    payoffs = np.maximum(terminal_prices - K, 0) * survival_indicator
    
    discounted_payoffs = np.exp(-r * T) * payoffs
    
    price = np.mean(discounted_payoffs)
    std_err = np.std(discounted_payoffs) / np.sqrt(len(discounted_payoffs))
    return price, std_err

Barrier_level = 120.0
barrier_price, barrier_err = price_up_and_out_call(paths, K, Barrier_level, T, r)

print(f"Up-and-Out Call Price (B={Barrier_level}): {barrier_price:.4f} (Std Err: {barrier_err:.4f})")

# European Call benchmark for comparison
euro_payoffs = np.maximum(paths[-1, :] - K, 0)
euro_price = np.mean(np.exp(-r * T) * euro_payoffs)
print(f"Standard European Call Price: {euro_price:.4f}")
print("\nNotice how exotic features generally reduce the option premium:")
print("Asian < European (due to volatility smoothing)")
print("Up-and-Out < European (due to knockout risk)")